In [1]:
# Imports and setup
import os
import json
from collections import defaultdict, Counter
from pathlib import Path
from tqdm import tqdm

DATASET_BASE = "/workspace/dataset"
COLLECTIONS = ["lastfm", "suno", "udio"]
OUTPUT_DIR = "/workspace/ngram_datasets"

# Distinctness gate for window retention:
#   True  = legacy behavior: trigrams kept only if all 3 chords are distinct,
#           tetragrams if at least 3 of 4 are, compared case-insensitively
#           (IV and iv count as one). Exports to the legacy filenames.
#   False = repetition-friendly mode: windows are kept after consecutive
#           de-duplication (so I IV I IV survives), but windows built from a
#           single root are dropped (I i I, iv IV iv IV, etc., where only
#           mode/case differs). Exports to filenames with the suffix _gate_false.
APPLY_DISTINCTNESS_GATE = False

print("Imports loaded | APPLY_DISTINCTNESS_GATE =", APPLY_DISTINCTNESS_GATE)

Imports loaded | APPLY_DISTINCTNESS_GATE = False


In [2]:
# --- Core functions ---
import re

# ═══════════════════════════════════════════════════════════════════════════════
# 12-TONE CHORD NORMALISER (identical to 05_axe_chords.ipynb — keep in sync)
# Resolves enharmonic spellings from the music21 Roman-numeral conversion to a
# canonical 12-tone vocabulary before any n-gram is formed, e.g. #V → bVI,
# #VII → I, ##IV → V, biv → iii, BVII → bVII. Case (chord quality) is preserved.
# ═══════════════════════════════════════════════════════════════════════════════
_DEG_SEMI = {'I': 0, 'II': 2, 'III': 4, 'IV': 5, 'V': 7, 'VI': 9, 'VII': 11}
_SEMI_MAJ = {0:'I', 1:'bII', 2:'II', 3:'bIII', 4:'III', 5:'IV',
             6:'#IV', 7:'V', 8:'bVI', 9:'VI', 10:'bVII', 11:'VII'}
_SEMI_MIN = {0:'i', 1:'bii', 2:'ii', 3:'biii', 4:'iii', 5:'iv',
             6:'#iv', 7:'v', 8:'bvi', 9:'vi', 10:'bvii', 11:'vii'}
_CHORD_RE = re.compile(
    r'^([#b]*)(VII|VI|V|IV|III|II|I|vii|vi|v|iv|iii|ii|i)$')

def normalize_chord(ch):
    """Resolve enharmonics via 12-tone semitone mapping."""
    norm = re.sub(r'^B(?=[IViv])', 'b', ch)
    m = _CHORD_RE.match(norm)
    if not m:
        return ch
    acc_str, deg = m.group(1), m.group(2)
    acc = acc_str.count('#') - acc_str.count('b')
    is_minor = deg[0].islower()
    semi = (_DEG_SEMI[deg.upper()] + acc) % 12
    return _SEMI_MIN[semi] if is_minor else _SEMI_MAJ[semi]

for _inp, _exp in [('#V','bVI'), ('#VII','I'), ('#III','IV'), ('##IV','V'),
                   ('biv','iii'), ('BVII','bVII'), ('bVI','bVI'), ('I','I'),
                   ('vi','vi'), ('#iv','#iv')]:
    assert normalize_chord(_inp) == _exp, (_inp, normalize_chord(_inp), _exp)


def parse_functional_lab(path: str) -> list:
    """Parse a _functional.lab file and return the list of functional labels
    (Roman numerals), enharmonically normalized via normalize_chord."""
    labels = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if len(parts) < 3:
                continue
            label = parts[2]
            if label == "N":
                continue
            labels.append(normalize_chord(label))
    return labels


def make_ngram(chords, n=3, apply_gate=True):
    """
    Create n-grams from chord sequence.
    - Remove consecutive duplicates first (labels arrive already normalized,
      so enharmonic duplicates like bVI followed by #V collapse here too).
    - apply_gate=True (legacy): trigrams (n=3) need all 3 chords different,
      tetragrams (n=4+) at least 3 different, compared case-insensitively.
    - apply_gate=False: windows are kept, except those built from a single
      root (case-insensitive), e.g. I i I or iv IV iv IV. Repetition across
      two or more roots (I IV I IV) survives.
    """
    if len(chords) < n:
        return []

    # Remove consecutive duplicates
    filtered = [chords[0]]
    for c in chords[1:]:
        if c != filtered[-1]:
            filtered.append(c)

    ngrams = []
    for i in range(len(filtered) - n + 1):
        ngram = tuple(filtered[i:i + n])
        unique = len(set(c.lower() for c in ngram))
        if apply_gate:
            if n == 3 and unique != 3:
                continue
            if n != 3 and unique < 3:
                continue
        else:
            # Single-root window: only mode/case alternation, no real motion
            if unique == 1:
                continue
        ngrams.append(ngram)
    return ngrams


# --- Genre normalization ---
target_genres = [
    'rock', 'pop', 'jazz', 'electronic', 'blues', 'metal', 'hiphop', 'country',
    'folk', 'soul', 'punk', 'classical', 'reggae', 'rnb', 'indie', 'funk',
    'dance', 'latin', 'ambient', 'experimental', 'world'
]

explicit_map = {
    'pink': 'pop', 'p!nk': 'pop', 'beatles': 'rock', 'the beatles': 'rock',
    'stones': 'rock', 'the stones': 'rock', 'alternative': 'rock', 'emo': 'punk',
    'shoegaze': 'rock', 'acoustic': 'folk', 'instrumental': 'other', 'soundtrack': 'other',
    'british': 'pop', '80s': 'pop', "80's": 'pop', '90s': 'rock', '90': 'rock',
    '70s': 'pop', '70': 'pop', 'raphiphop': 'hiphop', 'hip hop': 'hiphop', 'rap': 'hiphop',
    'indie rock': 'rock', 'indie': 'indie', 'funksoulrnb': 'soul', 'folkcountry': 'country',
}

def normalize_tag(tag):
    if tag is None:
        return 'other'
    t = tag.strip().lower()
    if t in explicit_map:
        return explicit_map[t]
    if t in target_genres:
        return t
    for genre in target_genres:
        if genre in t:
            return genre
    return 'other'


def extract_styles(style_json_path):
    """Extract and normalize styles from style JSON file."""
    with open(style_json_path, "r") as f:
        data = json.load(f)

    original = data.get("original", None)
    normalized_original_styles = []
    if isinstance(original, list):
        for style in original:
            ns = normalize_tag(style)
            if ns != 'other':
                normalized_original_styles.append(ns)
    elif isinstance(original, str):
        for style in [s.strip() for s in original.split(",") if s.strip()]:
            ns = normalize_tag(style)
            if ns != 'other':
                normalized_original_styles.append(ns)

    genres = data.get("genre_dortmund", {})
    dortmund_genres = [{'genre': normalize_tag(g), 'confidence': float(c)} for g, c in genres.items()]
    return normalized_original_styles, dortmund_genres


print("Core functions defined (with 12-tone enharmonic normalization)")

Core functions defined (with 12-tone enharmonic normalization)


In [3]:
# --- Build n-gram dataset from _functional.lab files ---

def create_ngram_dataset(base_path=DATASET_BASE, collections=COLLECTIONS, n=3, apply_gate=True):
    """
    Build n-gram aggregated dataset by reading pre-computed _functional.lab files.
    No tonality estimation or Roman numeral conversion — just read and form n-grams.
    apply_gate controls the distinctness retention gate (see make_ngram).
    """
    collection_data = {}

    for collection in collections:
        print(f"\nProcessing: {collection} (gate={apply_gate})")
        collection_path = os.path.join(base_path, collection)
        if not os.path.exists(collection_path):
            print(f"  Warning: {collection_path} does not exist, skipping")
            continue

        ngram_data = defaultdict(lambda: {
            "count": 0,
            "human_styles": Counter(),
            "dortmund_genres": Counter(),
            "song_ids": []
        })

        song_dirs = [d for d in os.listdir(collection_path)
                     if os.path.isdir(os.path.join(collection_path, d))]

        processed = 0
        skipped_no_file = 0
        skipped_empty = 0

        for song_id in tqdm(song_dirs, desc=f"  {collection}"):
            song_path = os.path.join(collection_path, song_id)
            func_lab = os.path.join(song_path, f"{song_id}_functional.lab")

            if not os.path.exists(func_lab):
                skipped_no_file += 1
                continue

            labels = parse_functional_lab(func_lab)
            if not labels:
                skipped_empty += 1
                continue

            ngrams = make_ngram(labels, n, apply_gate=apply_gate)
            if not ngrams:
                skipped_empty += 1
                continue

            processed += 1

            # Load styles (optional)
            style_json = os.path.join(song_path, f"{song_id}_style.json")
            original_styles, dortmund_genres = [], []
            if os.path.exists(style_json):
                try:
                    original_styles, dortmund_genres = extract_styles(style_json)
                except Exception:
                    pass

            for ngram in ngrams:
                key = tuple(ngram)
                ngram_data[key]["count"] += 1
                if song_id not in ngram_data[key]["song_ids"]:
                    ngram_data[key]["song_ids"].append(song_id)
                for style in original_styles:
                    ngram_data[key]["human_styles"][style] += 1
                for gi in dortmund_genres:
                    ngram_data[key]["dortmund_genres"][gi['genre']] += gi['confidence']

        collection_data[collection] = dict(ngram_data)
        print(f"  {len(ngram_data):,} unique {n}-grams | {processed:,} songs processed | "
              f"{skipped_no_file:,} no functional.lab | {skipped_empty:,} empty/no ngrams")

    return collection_data


print("Dataset builder defined")

Dataset builder defined


In [4]:
# --- Export function ---

def export_ngram_dataset(collection_data, n, output_dir=OUTPUT_DIR, apply_gate=True):
    """Export n-gram dataset to JSON files (same format as before).
    apply_gate=True writes the legacy filenames; False appends _gate_false."""
    subdir = {3: "trigrams", 4: "tetragrams"}.get(n, f"{n}grams")
    output_path = os.path.join(output_dir, subdir)
    os.makedirs(output_path, exist_ok=True)

    suffix = "" if apply_gate else "_gate_false"

    for collection, ngram_data in collection_data.items():
        export_data = []
        for ngram_tuple, stats in ngram_data.items():
            export_data.append({
                "ngram": list(ngram_tuple),
                "count": stats["count"],
                "human_styles": stats["human_styles"].most_common(10),
                "dortmund_genres": sorted(stats["dortmund_genres"].items(), key=lambda x: x[1], reverse=True),
                "song_ids": stats["song_ids"],
                "num_songs": len(stats["song_ids"])
            })
        export_data.sort(key=lambda x: x["count"], reverse=True)

        output_file = os.path.join(output_path, f"{n}gram_dataset_{collection}_ace{suffix}.json")
        with open(output_file, 'w') as f:
            json.dump(export_data, f, indent=2)
        print(f"Exported {len(export_data):,} {n}-grams for {collection} → {output_file}")


print("Export function defined")

Export function defined


In [5]:
# --- Run: Create and export trigram + tetragram datasets ---
# Gate is set once at the top of the notebook (APPLY_DISTINCTNESS_GATE).

for N in (3, 4):
    ngram_data_all = create_ngram_dataset(DATASET_BASE, COLLECTIONS, n=N,
                                          apply_gate=APPLY_DISTINCTNESS_GATE)

    print("\n" + "=" * 80)
    print(f"EXPORTING {N}-GRAM DATASETS (gate={APPLY_DISTINCTNESS_GATE})")
    print("=" * 80)
    export_ngram_dataset(ngram_data_all, n=N, apply_gate=APPLY_DISTINCTNESS_GATE)

    # Summary
    print("\n" + "=" * 80)
    print(f"{N}-GRAM STATISTICS (gate={APPLY_DISTINCTNESS_GATE})")
    print("=" * 80)
    for collection, ngram_data in ngram_data_all.items():
        total = sum(s["count"] for s in ngram_data.values())
        unique = len(ngram_data)
        print(f"\n{collection.upper()}:")
        print(f"  Unique {N}-grams: {unique:,}")
        print(f"  Total instances: {total:,}")
        if ngram_data:
            top_key = max(ngram_data, key=lambda k: ngram_data[k]["count"])
            top = ngram_data[top_key]
            print(f"  Most common: {' - '.join(top_key)} (count: {top['count']:,})")


Processing: lastfm (gate=False)


  lastfm: 100%|██████████| 19913/19913 [00:32<00:00, 612.29it/s]


  11,550 unique 3-grams | 18,781 songs processed | 997 no functional.lab | 135 empty/no ngrams

Processing: suno (gate=False)


  suno: 100%|██████████| 19972/19972 [00:34<00:00, 579.14it/s]


  9,161 unique 3-grams | 18,861 songs processed | 994 no functional.lab | 117 empty/no ngrams

Processing: udio (gate=False)


...

  92,167 unique 4-grams | 18,721 songs processed | 997 no functional.lab | 195 empty/no ngrams

Processing: suno (gate=False)


  suno: 100%|██████████| 19972/19972 [00:24<00:00, 802.96it/s]


  54,928 unique 4-grams | 18,794 songs processed | 994 no functional.lab | 184 empty/no ngrams

Processing: udio (gate=False)


  udio: 100%|██████████| 19992/19992 [00:16<00:00, 1188.09it/s]


  76,145 unique 4-grams | 18,663 songs processed | 961 no functional.lab | 368 empty/no ngrams

EXPORTING 4-GRAM DATASETS (gate=False)
Exported 92,167 4-grams for lastfm → /workspace/ngram_datasets/tetragrams/4gram_dataset_lastfm_ace_gate_false.json
Exported 54,928 4-grams for suno → /workspace/ngram_datasets/tetragrams/4gram_dataset_suno_ace_gate_false.json
Exported 76,145 4-grams for udio → /workspace/ngram_datasets/tetragrams/4gram_dataset_udio_ace_gate_false.json

4-GRAM STATISTICS (gate=False)

LASTFM:
  Unique 4-grams: 92,167
  Total instances: 1,396,018
  Most common: I - IV - I - IV (count: 17,630)

SUNO:
  Unique 4-grams: 54,928
  Total instances: 1,372,615
  Most common: vi - IV - I - V (count: 18,314)

UDIO:
  Unique 4-grams: 76,145
  Total instances: 917,453
  Most common: I - IV - I - IV (count: 9,280)
